### Transform Sprints Data

1. Read bronze_sprints table
2. Keep only the columns required for analytics (Drop url column)
3. Standardise column names using snake_case (constructorId → constructor_id, driverId → driver_id, raceName → race_name, positionText → finish_position_text)
4. Rename columns to make them more meaningful (date → race_date, grid → grid_position, laps → completed_laps, number → car_number, position → finish_position)
5. Filter out rows where season, round, constructor_id, or driver_id is null (business key validation)
6. Remove duplicate records
7. Transform values of column race_name to Title Case
8. Write the transformed data to silver_sprints table

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-configuration

In [0]:
%run ../00-common/03.Silver-Helper

In [0]:
bronze_table = F"{catalog_name}.{bronze_schema}.sprints"
silver_table = F"{catalog_name}.{silver_schema}.sprints"

In [0]:
from pyspark.sql import functions as F

In [0]:
sprints_df = (
    spark.read.table(bronze_table).filter((F.col("batch_id") == v_batch_id))
)

In [0]:
from pyspark.sql import functions as F

In [0]:
sprints_df = (
    spark.table(bronze_table)
        .filter((F.col("batch_id") == v_batch_id))
        .select("date",
                "raceName",
                "round",
                "season",
                "constructorId",
                "driverId",
                "grid",
                "laps",
                "number",
                "points",
                "position",
                "positionText",
                "status",
                "timestamp",
                "source_file")
        .withColumnsRenamed({
                "constructorId": "constructor_id",
                "driverId": "driver_id",
                "raceName": "race_name",
                "positionText": "finish_position_text",
                "date": "race_date",
                "grid": "grid_position",
                "laps": "completed_laps",
                "number": "car_number",
                "position": "finish_position",
                })
) 


### Standardise column names using snake_case (constructorId → constructor_id, driverId → driver_id, raceName → race_name, positionText → finish_position_text)
### Rename columns to make them more meaningful (date → race_date, grid → grid_position, laps → completed_laps, number → car_number, position → finish_position)

In [0]:
display(sprints_df)

### Filter out rows where season, round, constructor_id, or driver_id is null (business key validation)

In [0]:
sprints_valid_df =(
    sprints_df
        .filter(
            F.col("season").isNotNull() &
            F.col("round").isNotNull() &
            F.col("constructor_id").isNotNull() &
            F.col("driver_id").isNotNull()
        )    
)        

In [0]:
display(sprints_valid_df)

### Remove duplicate records

In [0]:
sprints_distinct_df = (
    sprints_valid_df
        .dropDuplicates(["season", "round", "driver_id", "constructor_id"])
)

In [0]:
display(sprints_valid_df.count()-sprints_distinct_df.count())

### Transform values of column race_name to Title Case

In [0]:
sprints_final_df = (
    sprints_distinct_df
        .withColumn("race_name", F.initcap(F.col("race_name")))
)

In [0]:
display(sprints_final_df)

In [0]:
write_to_silver(
    input_df = sprints_final_df,
    target_table = silver_table,
    merge_condition= "t.season = s.season AND t.round = s.round AND t.driver_id = s.driver_id AND t.constructor_id = s.constructor_id",
    columns_to_update=[
        "race_name",
        "race_date",
        "grid_position",
        "completed_laps",
        "car_number",
        "points",
        "final_position",
        "final_position_text",
        "status",
        "timestamp",
        "source_file",
        "batch_id"
    ]
)



In [0]:
display(spark.table(silver_table))
